[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/05_cnn/05_cnn.ipynb)

# 05. Convolutional Neural Networks (CNN)

> 원본 강의: [Lec 11, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — Convolution 레이어, Max Pooling, CNN으로 MNIST 99%+

## 이론

### 1) 왜 이미지에는 CNN인가
04번 노트북의 MLP는 이미지를 1차원으로 펼쳐서(`Flatten`) 넣었기 때문에, 픽셀 간의 **공간적 구조(주변 픽셀과의 관계)**를 잃어버립니다. CNN은 이미지를 2차원 그대로 처리하며 지역적인 패턴(모서리, 곡선 등)을 학습합니다.

### 2) Convolution(합성곱) 레이어
작은 크기의 **필터(커널)**가 이미지 위를 슬라이딩하며 지역적인 특징을 추출해 **Feature Map**을 만듭니다.
- **Stride**: 필터가 이동하는 간격
- **Padding**: 가장자리 정보 손실을 막기 위해 테두리에 0을 채우는 것
- 필터의 가중치는 이미지 전체에서 **공유(shared weights)**되므로, MLP보다 파라미터 수가 훨씬 적습니다.

### 3) Pooling(풀링) 레이어
Feature Map의 크기를 줄여(다운샘플링) 연산량을 줄이고, 작은 위치 변화에 강건하게 만듭니다. 가장 흔한 방식은 **Max Pooling**(구역 내 최댓값만 남김).

### 4) 전형적인 CNN 구조
```
[Conv -> ReLU -> Pooling] 를 여러 번 반복 -> Flatten -> Fully Connected -> 출력
```
앞부분(Conv+Pooling)이 특징을 추출하고, 뒷부분(FC)이 그 특징으로 분류를 수행합니다.

## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 실습 1. 작은 이미지로 Convolution 동작 눈으로 확인하기

In [ ]:
import matplotlib.pyplot as plt

# 세로줄 이미지를 만들고, 수직 엣지를 검출하는 필터를 적용해본다
img = torch.zeros(1, 1, 8, 8)
img[:, :, :, 4:] = 1.0  # 오른쪽 절반이 밝은 이미지

vertical_edge_filter = torch.tensor([[[[-1., 0., 1.],
                                         [-1., 0., 1.],
                                         [-1., 0., 1.]]]])

conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    conv.weight.copy_(vertical_edge_filter)
feature_map = conv(img)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img[0, 0], cmap="gray")
axes[0].set_title("입력 이미지 (8x8)")
axes[1].imshow(feature_map[0, 0].detach(), cmap="gray")
axes[1].set_title("Convolution 결과 (Feature Map)\n경계선이 강조됨")
plt.show()

## 실습 2. CNN으로 MNIST 분류 (04번 MLP와 성능 비교)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 28x28 -> 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {n_params:,}")

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_ds)
    test_acc = evaluate(model, test_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  test_acc={test_acc:.4f}")

## 정리 & 연습 문제
- 같은 3 epoch 기준으로 04번 노트북의 MLP와 정확도를 비교해보세요 — 보통 CNN이 더 빠르게, 더 높은 정확도에 도달합니다.
- `Conv2d`의 채널 수(16, 32)나 층 수를 늘려보고 정확도/속도 변화를 관찰해보세요.
- 다음 노트북([06_rnn.ipynb](../06_rnn/06_rnn.ipynb))에서는 순서가 있는 데이터(시퀀스)를 다루는 RNN으로 넘어갑니다.